# ROUND2 EDA — Strategy-Focused Notebook

Round 2 should be less about generic order-book sightseeing and more about whether our quoting logic actually earns edge.

**Sections**

A. Market behavior

- mid
- spread
- vol
- depth
- imbalance

B. Execution quality

- markouts
- realized spread
- fill probability
- toxicity by regime

C. Strategy implications

- quote width rules
- skew rules
- inventory rules
- MAF valuation

**Top priorities from here**

1. Trade markout / adverse selection
2. Passive fill-quality proxy
3. Simple replay backtest with inventory
4. Signal decay by horizon
5. MAF value / break-even analysis


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

try:
    import seaborn as sns
except ImportError:
    sns = None

if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (13, 5)

CWD = Path.cwd().resolve()
if list(CWD.glob("prices_round_2_day_*.csv")):
    DATA = CWD
elif list((CWD.parent).glob("prices_round_2_day_*.csv")):
    DATA = CWD.parent
else:
    DATA = CWD / "ROUND2"

SHORT = {
    "ASH_COATED_OSMIUM": "ACO",
    "INTARIAN_PEPPER_ROOT": "IPR",
}

COLORS = {
    "ACO": "#0d3b66",
    "IPR": "#ee964b",
}

MARKOUT_HORIZONS = [1, 5, 10, 20, 50]
FILL_HORIZON = 2_000


In [ ]:
def draw_table_heatmap(ax, df: pd.DataFrame, title: str, cmap: str = "RdYlGn", center=None):
    plot_df = df.copy()
    values = plot_df.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        ax.set_title(title)
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.axis("off")
        return
    if center is None:
        vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
    else:
        span = float(np.nanmax(np.abs(finite - center)))
        vmin, vmax = center - span, center + span
    im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(plot_df.shape[1]))
    ax.set_xticklabels(plot_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(plot_df.shape[0]))
    ylabels = []
    for idx in plot_df.index:
        if isinstance(idx, tuple):
            ylabels.append(" | ".join(map(str, idx)))
        else:
            ylabels.append(str(idx))
    ax.set_yticklabels(ylabels)
    for i in range(plot_df.shape[0]):
        for j in range(plot_df.shape[1]):
            val = plot_df.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


def _parse_day(path: Path) -> int:
    m = re.search(r"day_(-?\d+)", path.stem)
    if not m:
        raise ValueError(f"Could not parse day from {path}")
    return int(m.group(1))


def load_prices(folder: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(folder.glob("prices_round_2_day_*.csv")):
        df = pd.read_csv(path, sep=";")
        df["day"] = _parse_day(path)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out["t"] = out["day"] * 1_000_000 + out["timestamp"]
    out = out.sort_values(["product", "t"]).reset_index(drop=True)
    return out


def load_trades(folder: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(folder.glob("trades_round_2_day_*.csv")):
        df = pd.read_csv(path, sep=";")
        df["day"] = _parse_day(path)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out = out.rename(columns={"symbol": "product"})
    out["t"] = out["day"] * 1_000_000 + out["timestamp"]
    out = out.sort_values(["product", "t"]).reset_index(drop=True)
    return out


def enrich_prices(prices: pd.DataFrame) -> pd.DataFrame:
    p = prices.copy()
    p["spread"] = p["ask_price_1"] - p["bid_price_1"]
    p["depth_top"] = p[["bid_volume_1", "ask_volume_1"]].fillna(0).sum(axis=1)
    p["depth_3"] = p[[
        "bid_volume_1", "bid_volume_2", "bid_volume_3",
        "ask_volume_1", "ask_volume_2", "ask_volume_3",
    ]].fillna(0).sum(axis=1)
    top_total = p["bid_volume_1"].fillna(0) + p["ask_volume_1"].fillna(0)
    p["imb1"] = np.where(
        top_total > 0,
        (p["bid_volume_1"].fillna(0) - p["ask_volume_1"].fillna(0)) / top_total,
        np.nan,
    )
    denom = p["bid_volume_1"].fillna(0) + p["ask_volume_1"].fillna(0)
    p["microprice"] = np.where(
        denom > 0,
        (
            p["ask_price_1"] * p["bid_volume_1"].fillna(0)
            + p["bid_price_1"] * p["ask_volume_1"].fillna(0)
        ) / denom,
        p["mid_price"],
    )
    p["micro_minus_mid_bps"] = (p["microprice"] - p["mid_price"]) / p["mid_price"] * 1e4
    p["ret_1_bps"] = p.groupby("product")["mid_price"].transform(lambda s: np.log(s).diff() * 1e4)
    p["vol_50_bps"] = p.groupby("product")["ret_1_bps"].transform(lambda s: s.rolling(50).std())
    for h in MARKOUT_HORIZONS:
        p[f"mid_fwd_{h}"] = p.groupby("product")["mid_price"].shift(-h)
        p[f"ret_fwd_{h}_bps"] = (
            (p[f"mid_fwd_{h}"] - p["mid_price"]) / p["mid_price"] * 1e4
        )
    return p


def attach_prevailing_book(trades: pd.DataFrame, prices: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "product", "t", "mid_price", "bid_price_1", "ask_price_1", "spread",
        "imb1", "vol_50_bps", "depth_top", "depth_3",
    ] + [f"mid_fwd_{h}" for h in MARKOUT_HORIZONS]

    out = []
    for product, tp in trades.groupby("product", sort=False):
        pp = prices.loc[prices["product"] == product, cols].sort_values("t")
        merged = pd.merge_asof(
            tp.sort_values("t"),
            pp,
            on="t",
            by="product",
            direction="backward",
        )
        out.append(merged)
    merged = pd.concat(out, ignore_index=True)
    merged["trade_sign"] = np.sign(merged["price"] - merged["mid_price"])
    merged.loc[merged["trade_sign"] == 0, "trade_sign"] = np.nan
    merged["passive_side"] = -merged["trade_sign"]
    merged["edge_at_fill_bps"] = merged["passive_side"] * (merged["price"] - merged["mid_price"]) / merged["mid_price"] * 1e4
    for h in MARKOUT_HORIZONS:
        merged[f"markout_{h}_bps"] = merged["passive_side"] * (merged[f"mid_fwd_{h}"] - merged["price"]) / merged["mid_price"] * 1e4
        merged[f"realized_spread_{h}_bps"] = 2 * merged["passive_side"] * (merged["price"] - merged[f"mid_fwd_{h}"]) / merged["mid_price"] * 1e4
    return merged


def bucket_series(s: pd.Series, q: int = 5) -> pd.Series:
    ranked = s.rank(method="first")
    return pd.qcut(ranked, q=q, labels=[f"Q{i}" for i in range(1, q + 1)])


prices = enrich_prices(load_prices(DATA))
trades = load_trades(DATA)
trade_book = attach_prevailing_book(trades, prices)

prices["short"] = prices["product"].map(SHORT)
trade_book["short"] = trade_book["product"].map(SHORT)
trade_book["spread_bucket"] = trade_book.groupby("product")["spread"].transform(bucket_series)
trade_book["vol_bucket"] = trade_book.groupby("product")["vol_50_bps"].transform(bucket_series)
trade_book["imb_bucket"] = pd.cut(
    trade_book["imb1"],
    bins=[-1.01, -0.6, -0.2, 0.2, 0.6, 1.01],
    labels=["sell-heavy", "lean-sell", "balanced", "lean-buy", "buy-heavy"],
)

prices.head()

## A. Market behavior

Start with the state variables we might quote off of: mid, spread, realized volatility, displayed depth, and top-of-book imbalance.

In [ ]:
day_bounds = sorted(prices["day"].unique())
day_lines = [(d - min(day_bounds)) * 1_000_000 for d in day_bounds]

fig, axes = plt.subplots(5, 2, figsize=(15, 18), sharex="col")
metrics = [
    ("mid_price", "Mid"),
    ("spread", "Spread"),
    ("vol_50_bps", "Rolling vol (50 ticks, bps)"),
    ("depth_top", "Top depth"),
    ("imb1", "Imbalance"),
]

for col_ix, product in enumerate(sorted(prices["product"].unique())):
    sub = prices.loc[prices["product"] == product].copy()
    short = SHORT[product]
    for row_ix, (metric, title) in enumerate(metrics):
        ax = axes[row_ix, col_ix]
        ax.plot(sub["t"], sub[metric], lw=0.9, color=COLORS[short])
        for x in day_lines:
            ax.axvline(x, ls="--", lw=0.7, color="gray", alpha=0.35)
        ax.set_title(f"{short} — {title}")
        if row_ix == len(metrics) - 1:
            ax.set_xlabel("Global time")
plt.tight_layout()

market_summary = (
    prices.groupby("short")[["spread", "ret_1_bps", "vol_50_bps", "depth_top", "imb1"]]
    .agg(["mean", "median", "std"])
    .round(3)
)
market_summary

## B. Execution quality

This is the section that converts trade prints into quoting diagnostics. We use prevailing mid/book state, then ask whether a passive fill would have been good or toxic.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

markout_col = "markout_10_bps"
rs_col = "realized_spread_10_bps"

trade_book.boxplot(column=markout_col, by="short", ax=axes[0, 0])
axes[0, 0].set_title("10-tick passive markout")
axes[0, 0].set_xlabel("")

trade_book.boxplot(column=rs_col, by="short", ax=axes[0, 1])
axes[0, 1].set_title("10-tick realized spread")
axes[0, 1].set_xlabel("")

markout_curve = (
    trade_book.groupby("short")[[f"markout_{h}_bps" for h in MARKOUT_HORIZONS]]
    .mean()
    .T
)
markout_curve.index = MARKOUT_HORIZONS
markout_curve.plot(ax=axes[1, 0], marker="o")
axes[1, 0].set_title("Signal decay by horizon")
axes[1, 0].set_xlabel("Horizon (ticks)")
axes[1, 0].set_ylabel("Mean passive markout (bps)")

toxicity = (
    trade_book.groupby(["short", "spread_bucket", "imb_bucket"])[markout_col]
    .mean()
    .unstack("imb_bucket")
)
draw_table_heatmap(axes[1, 1], toxicity, "10-tick markout by spread / imbalance regime", cmap="RdYlGn", center=0)

plt.tight_layout()

execution_summary = (
    trade_book.groupby("short")[["edge_at_fill_bps", markout_col, rs_col]]
    .agg(["mean", "median", "count"])
    .round(3)
)
execution_summary

In [ ]:
def estimate_fill_proxy(prices: pd.DataFrame, trades: pd.DataFrame, horizon: int = FILL_HORIZON) -> pd.DataFrame:
    rows = []
    for product, book in prices.groupby("product"):
        tape = trades.loc[trades["product"] == product, ["t", "price", "quantity"]].sort_values("t")
        trade_t = tape["t"].to_numpy()
        trade_px = tape["price"].to_numpy()
        for row in book[["t", "spread", "imb1", "vol_50_bps", "bid_price_1", "ask_price_1"]].itertuples(index=False):
            lo = np.searchsorted(trade_t, row.t, side="right")
            hi = np.searchsorted(trade_t, row.t + horizon, side="right")
            future_px = trade_px[lo:hi]
            bid_hit = np.any(future_px <= row.bid_price_1) if len(future_px) else False
            ask_hit = np.any(future_px >= row.ask_price_1) if len(future_px) else False
            rows.append({
                "product": product,
                "spread": row.spread,
                "imb1": row.imb1,
                "vol_50_bps": row.vol_50_bps,
                "bid_fill_proxy": float(bid_hit),
                "ask_fill_proxy": float(ask_hit),
            })
    out = pd.DataFrame(rows)
    out["short"] = out["product"].map(SHORT)
    out["spread_bucket"] = out.groupby("product")["spread"].transform(bucket_series)
    out["vol_bucket"] = out.groupby("product")["vol_50_bps"].transform(bucket_series)
    out["imb_bucket"] = pd.cut(
        out["imb1"],
        bins=[-1.01, -0.6, -0.2, 0.2, 0.6, 1.01],
        labels=["sell-heavy", "lean-sell", "balanced", "lean-buy", "buy-heavy"],
    )
    out["either_fill_proxy"] = ((out["bid_fill_proxy"] + out["ask_fill_proxy"]) > 0).astype(float)
    return out


fill_proxy = estimate_fill_proxy(prices, trades)

fill_table = (
    fill_proxy.groupby(["short", "spread_bucket", "imb_bucket"])["either_fill_proxy"]
    .mean()
    .unstack("imb_bucket")
    .round(3)
)

fig, ax = plt.subplots(figsize=(12, 5))
draw_table_heatmap(ax, fill_table, f"Passive fill proxy within {FILL_HORIZON} time units", cmap="Blues")
plt.tight_layout()
plt.show()

fill_table

## C. Strategy implications

This section turns the diagnostics into concrete knob-setting: when to widen, when to skew, and how hard to lean against inventory.

In [ ]:
width_rules = (
    trade_book.groupby(["short", "spread_bucket"])
    .agg(
        mean_spread=("spread", "mean"),
        markout_10_bps=("markout_10_bps", "mean"),
        realized_spread_10_bps=("realized_spread_10_bps", "mean"),
    )
    .round(3)
)

skew_rules = (
    prices.groupby(["short", pd.cut(prices["imb1"], bins=[-1.01, -0.6, -0.2, 0.2, 0.6, 1.01], labels=["sell-heavy", "lean-sell", "balanced", "lean-buy", "buy-heavy"])])
    [["ret_fwd_1_bps", "ret_fwd_5_bps", "ret_fwd_10_bps"]]
    .mean()
    .round(3)
)

inventory_rules = (
    prices.groupby(["short", pd.cut(prices["vol_50_bps"], bins=5)])[["ret_1_bps", "spread", "depth_top"]]
    .agg(["mean", "std", "median"])
    .round(3)
)

display(width_rules)
display(skew_rules)
display(inventory_rules)


In [ ]:
# Parameterized MAF / extra quote capacity break-even view.
# Keep the label generic for now; plug in the exact round-2 mechanism once defined.

maf_grid = pd.DataFrame(
    [
        {
            "extra_fill_prob": fp,
            "avg_fill_size": size,
            "edge_bps": edge,
            "expected_value_per_quote": fp * size * edge / 1e4,
        }
        for fp in [0.01, 0.03, 0.05, 0.10]
        for size in [2, 5, 10]
        for edge in [0.5, 1.0, 2.0, 3.0]
    ]
)

maf_pivot = maf_grid.pivot_table(
    index=["extra_fill_prob", "avg_fill_size"],
    columns="edge_bps",
    values="expected_value_per_quote",
)
maf_pivot.style.format("{:.6f}")


## Next extension

The next notebook upgrade should be a lightweight replay backtest with inventory state:

- quote both sides from a fair price
- widen by spread / vol regime
- skew by imbalance or microprice signal
- clip quoting when inventory breaches limits
- compare realized spread vs adverse selection across parameter sets

That is the fastest way to move from diagnostics to an actual round 2 submission loop.